# Ten real FeTA fetal brains on nine matched growth charts

This local meeting notebook uses FeTA 2.2 SVR images and expert segmentations, not synthetic data. It fits only QC-passing cases labeled `Neurotypical`, then displays five neurotypical and five pathological examples. A reference flag is a research screen, not a diagnosis. FeTA data and derived images remain subject to FeTA access terms and are not committed.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display
from fetal_brain_growth.feta_gallery import build_feta_gallery
from fetal_brain_growth.feta_reference import resolve_feta_root

In [ ]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
FETA_ROOT = resolve_feta_root()  # or resolve_feta_root('/path/to/feta_2.2')
OUT = ROOT/'meeting_outputs/feta_10_cases_matched'
if not (OUT/'case_summary.csv').exists():
    paths = build_feta_gallery(FETA_ROOT, OUT, reference='feta-neurotypical', feta_degree=2)
else:
    paths = {
        'summary': OUT/'case_summary.csv',
        'scores': OUT/'reference_scores.csv',
        'overview': OUT/'ten_case_overview.png',
        'growth_chart': OUT/'ten_case_growth_chart.png',
        'metadata': OUT/'reference_metadata.json',
    }
print('FeTA root:', FETA_ROOT)
print('Meeting outputs:', OUT)

## Cohort selection and independent labels

The gallery deliberately includes both FeTA phenotype groups. Phenotype is supplied by the dataset; the volumetric screen is calculated here. A pathological phenotype may have all volumes within range, and a reference flag alone does not establish pathology.

In [ ]:
summary = pd.read_csv(paths['summary'])
display(summary[['subject_id','gestational_age_weeks','feta_phenotype','volume_screen','reference_result_detail']])

## Expert segmentations in standard orientation

In [ ]:
display(Image(filename=str(paths['overview']), width=1500))

## Nine exact-label volume charts

The panels show total brain, intracranial volume, external CSF, cortical gray matter, white matter, ventricles, cerebellum, deep gray matter, and brainstem with P3/P10/P25/P50/P75/P90/P97 bands. Red points fall outside P3–P97.

In [ ]:
display(Image(filename=str(paths['growth_chart']), width=1500))

## Flagged measurements and per-case report

In [ ]:
scores = pd.read_csv(paths['scores'])
flagged = scores[scores.status.isin(['low_reference_flag','high_reference_flag'])]
display(flagged[['subject_id','gestational_age_weeks','region','volume_ml','estimated_percentile_bounded','status']])
display(Image(filename=str(OUT/'case_cards/sub-050_case_report.png'), width=1500))

## How the reference was generated

For each measure, the default model is quadratic in centered gestational age with log-volume as the response. Quantiles are `exp(fitted_log_volume + Normal_quantile × residual_SD)`. No biological-volume outliers are removed. Cubic fitting is available with `fbg feta-reference --degree 3`, but should be treated as a sensitivity analysis for this small cohort.

In [ ]:
metadata = json.loads(Path(paths['metadata']).read_text())
print({key: metadata[key] for key in [
    'subjects','age_min_weeks','age_max_weeks','degree','quantiles','segmentation_qc_excluded_cases']})
diagnostics = pd.DataFrame(metadata['diagnostics']).T.reset_index(names='region')
display(diagnostics[['region','log_volume_r_squared','leave_one_out_rmse_log_volume','log_residual_sd']].style.format({
    'log_volume_r_squared': '{:.3f}',
    'leave_one_out_rmse_log_volume': '{:.3f}',
    'log_residual_sd': '{:.3f}',
}))

## Interpretation limits

- The FeTA control set is small, cross-sectional, and partly reused as displayed normal examples.
- Do not extrapolate outside 22.7–34.8 weeks.
- Ventricular, deep-gray, and brainstem fits are especially uncertain; inspect diagnostics.
- Review the complete 3-D image and segmentation, gestational-age uncertainty, morphology, and clinical context with a fetal neuroradiologist.
- A larger independent cohort processed with the same frozen pipeline is required for clinical validation.